# Chapter 4 — Pointer Fundamentals

Part 2: Pointer arithmetic and memory operations.

**Thread.**
- A logical index becomes an element offset through `shape` and `stride`.
- An element offset becomes a byte address through `itemsize`.
- A byte address becomes a typed pointer expression in a Triton kernel.
- The kernel receives only that pointer, so layout is the host's responsibility.
- The chapter follows the chain from paper/PyTorch to 1-D and 2-D kernels to stride-aware kernels.

**Ladder.**
- **Method 01:** address arithmetic on paper and in PyTorch.
- **Method 02:** Triton kernels built from `base + element offset`.
- **Method 03:** stride-aware kernels for non-contiguous views.

**Study tip.**
- Take a `(3,4)` `float32` matrix.
- Hand-compute the address of `(2,3)` in elements: `2*4 + 3 = 11`.
- Convert it to bytes: `11*4 = 44`.
- Repeat for the transpose, stride `(1,4)`, and the slice `A[:, 1:3]`, `storage_offset = 1`.
- If those three numbers match the next cells, the rest of the chapter is bookkeeping.

**Learning objectives**
- Read a kernel argument such as `x_ptr` as a **typed base pointer** and write `addr = base + offset`.
- Convert an element offset into a byte address with `itemsize`, and name the itemsize of fp32, fp16, int8, and int64.
- Derive row-major and column-major offsets from `shape` and `stride`.
- Read `shape`, `stride()`, `storage_offset()`, and `data_ptr()` from a PyTorch tensor and map a logical index to storage.
- Write 1-D and 2-D Triton kernels with explicit offsets, including an `int64` path for tensors with more than `2**31` elements.
- Pass strides as runtime arguments and run a stride-aware kernel on a transposed and a sliced tensor.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../../part1_foundations/ch03_vector_construction/03_complete.ipynb) · [Next →](../ch05_load_store/05_complete.ipynb)

**Estimated time:** 60–75 min · **Difficulty:** Intermediate

**Prerequisites:** Chapter 3 (vector construction, `tl.arange`, `tl.constexpr`); basic linear indexing; familiarity with `torch.Tensor.shape`, `dtype`, and `stride()`.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

### Base pointer + element offset

```text
x_ptr ──► [ x0 ][ x1 ][ x2 ][ x3 ][ x4 ][ x5 ]
element:      0      1      2      3      4      5
x_ptr + 3 ────────────────► x3
```

- `x_ptr` is not a number in the kernel source; it is a typed pointer.
- `x_ptr + 3` means three **elements** past the base, so the compiler multiplies by the element size.

### Element offset vs byte offset

```text
float32, itemsize = 4 bytes

element offset:   0     1     2     3
byte offset:      0     4     8    12
address:        base  base+4 base+8 base+12
```

- The Triton expression `x_ptr + 3` uses the element row.
- A raw integer such as `x.data_ptr() + 12` uses the byte row.

### Row-major vs column-major stride walk

```text
row-major, shape (3,4), stride (4,1)
storage:  0  1  2  3 | 4  5  6  7 | 8  9 10 11
logical: (0,0)(0,1)(0,2)(0,3)(1,0) ... (2,3)
walk: row 0 left→right, then row 1, then row 2

column-major, shape (3,4), stride (1,3)
storage:  0  1  2 | 3  4  5 | 6  7  8 | 9 10 11
logical: (0,0)(1,0)(2,0)(0,1)(1,1) ... (2,3)
walk: column 0 top→bottom, then column 1, then column 2
```

- Row-major walks a row at a time; column-major walks a column at a time.
- The same logical matrix can occupy the same storage in either order.

### The 2-D address formula

```text
off(i, j) = i * stride0 + j * stride1

row-major:    off = i*4 + j      because stride = (4, 1)
column-major: off = i*1 + j*3    because stride = (1, 3)

addr_elem = base + off
addr_byte = base_byte + itemsize * off
```

- The offset is one dot product between the index pair and the stride pair.
- The byte address is one more multiplication by `itemsize`.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu, cdiv

print(f"device: {DEVICE} | GPU: {HAS_GPU} | torch: {torch.__version__} | triton: {triton.__version__}")

## Part I — Mathematics/Concepts

- **Connection.** Builds on Chapter 3's tiles: this part supplies the addresses those tiles operate on.
- **Thread.** The chain is logical index → element offset → byte address → pointer expression.
- **Rule.** The element offset is what a Triton kernel adds to a pointer.
- **Where.** The byte address is what `data_ptr()` and external tools report.
- **Rule.** The stride converts a logical index into an element offset.
- **Bridge.** Keep the three objects separate and the twelve sections collapse into one rule.

### 4.1 Pointer arguments

- **Connection.** Builds on Chapter 3's `tl.arange` tiles: a pointer is the missing address half of a tile.
- **Rule.** A kernel parameter such as `x_ptr` is a typed pointer to the first element of the buffer the host passed; shape, stride, and dtype are not kernel metadata.
- **Formula.** $\text{addr}(x,i)=\text{base}(x)+\text{off}(i)$, and `x_ptr + k` advances $k$ elements, so $4k$ bytes for `float32` and $2k$ bytes for `float16`.
- **Example.** For `x = torch.arange(6, dtype=torch.float32)`, `x_ptr + 5` is element 5, which is $5\cdot4=20$ bytes past the base; the next code cell asserts the shape, stride, itemsize, and all six byte offsets.
- **Where.** Raw pointers let one kernel serve any shape and any layout; every later chapter uses `tl.load(x_ptr + offs)`.
- **Triton.** Use `x_ptr` directly in `tl.load` and `tl.store`; do not cast it to an integer unless you are deliberately doing byte-level work.
- **Failure.** Passing a non-contiguous tensor and assuming `offs = 0..numel-1` walks the logical order; it walks storage order.
- **Bridge.** If `base` is the first element of a view, §4.2 asks which element that is.

### 4.2 Base pointer

- **Connection.** Builds on §4.1: the base is the pointer argument, now read through a view.
- **Rule.** `base = x_ptr = data_ptr(x)`; for a view with `storage_offset = s`, $\text{data\_ptr}(x)=\text{storage\_base}+s\cdot\text{itemsize}$.
- **Formula.** $\text{base}=\text{data\_ptr}(x)$, so kernel offset `0` is the view's first logical element, not necessarily the allocation start.
- **Where.** This makes a sliced tensor usable without copying: pass the slice and its `data_ptr` already points at the first requested element.
- **Triton.** Do **not** add `storage_offset()` again inside the kernel; the host's `data_ptr` already did that.
- **Failure.** Double-counting the storage offset, or treating `data_ptr` as the allocation start, reads the wrong element by `s` positions.
- **Bridge.** Once the base is fixed, §4.3 adds offsets to it.

### 4.3 Pointer plus offset

- **Connection.** Builds on §4.1–§4.2: the base is fixed, and now an offset moves through the buffer.
- **Rule.** The offset may be a scalar (`x_ptr + 3`) or a tile (`x_ptr + offs`, where `offs` is a `tl.arange` vector); pointer-plus-tile produces a tile of pointers.
- **Formula.** $\text{addr}_i=\text{base}+\text{off}_i$ with $\text{off}_i=\text{pid}\cdot B+i$.
- **Example.** For a six-element `float32` vector, element offsets `0,1,2,3,4,5` correspond to byte offsets `0,4,8,12,16,20`; the next code cell asserts both rows.
- **Where.** This is the entire 1-D kernel skeleton: one program owns `BLOCK` consecutive elements and the tail is guarded by a mask.
- **Triton.**
```python
offs = pid * BLOCK + tl.arange(0, BLOCK)
mask = offs < n_elements
x = tl.load(x_ptr + offs, mask=mask, other=0.0)
```
- **Failure.** Confusing the element offset with a byte offset and pre-multiplying by the itemsize makes the kernel skip four elements for every one it should read.
- **Bridge.** §4.4 separates the two offset units explicitly.

### 4.4 Element offset vs byte offset

- **Connection.** Builds on §4.3's pointer-plus-offset; the only question is which units the offset uses.
- **Rule.** Triton pointer arithmetic is in **elements**; Python's `tensor.data_ptr()` and NumPy's `.strides` are in **bytes**.
- **Formula.** $\text{addr}_\text{byte}=\text{base}_\text{byte}+\text{itemsize}\cdot\text{off}_\text{elem}$.

| dtype | itemsize | offset 11 in bytes |
|---|---:|---:|
| `float64` / `int64` | 8 | 88 |
| `float32` / `int32` | 4 | 44 |
| `float16` / `bfloat16` | 2 | 22 |
| `int8` / `uint8` | 1 | 11 |

- **Example.** Element offset `11` is `44` bytes for `float32`, `22` for `float16`, and `11` for `int8`; the next code cell asserts the table and these conversions.
- **Where.** Diagnostics, profilers, and CUDA APIs speak bytes; kernels speak elements.
- **Triton.** If you have a byte offset `b`, convert once on the host with `b // x.element_size()` before passing it.
- **Failure.** Adding a byte offset to a Triton pointer, or an element offset to `data_ptr()`, is off by a factor of the itemsize.
- **Bridge.** §4.5 writes the element-offset rule for any number of dimensions.

### 4.5 Pointer arithmetic

- **Connection.** Builds on §4.1–§4.4: base, offset, and units now become one affine expression.
- **Rule.** Pointer arithmetic is **affine**: one base plus a linear combination of indices; there is no multiplication of pointers and no division.
- **Formula.** For a $d$-dimensional index, $\text{off}=\sum_{k=0}^{d-1} i_k s_k$ and $\text{addr}=\text{base}+\text{off}$.
- **Where.** Every indexing scheme in the book — row-major, column-major, tiled, strided, broadcast — is a choice of the coefficients $s_k$.
- **Triton.** Build the offset as a tile expression, then add it once: `x_ptr + offs`; reuse `offs` for the store.
- **Failure.** `x.data_ptr() + a*stride` is a byte address and cannot be passed back as a Triton pointer without a cast and a division.
- **Bridge.** The coefficients $s_k$ are the layout; §4.6 starts with the contiguous choice.

In [ ]:
def test_address_arithmetic() -> None:
    x = torch.arange(6, dtype=torch.float32)
    item = x.element_size()
    base = x.data_ptr()
    offs = torch.arange(6)
    byte_offs = offs * item

    assert tuple(x.shape) == (6,)
    assert x.stride() == (1,)
    assert item == 4
    assert offs.tolist() == [0, 1, 2, 3, 4, 5]
    assert byte_offs.tolist() == [0, 4, 8, 12, 16, 20]
    for i in range(6):
        assert x[i].data_ptr() == base + i * item
    assert x[5].data_ptr() == base + 5 * item == base + 20

    # The element-vs-byte table from §4.4.
    for dt, byte_offset in (
        (torch.float64, 88), (torch.float32, 44),
        (torch.float16, 22), (torch.bfloat16, 22),
        (torch.int32, 44), (torch.int8, 11),
    ):
        assert 11 * torch.empty(0, dtype=dt).element_size() == byte_offset
    print("address arithmetic: element/byte offsets asserted")


test_address_arithmetic()

### 4.6 Contiguous memory

- **Connection.** Builds on §4.5's coefficients $s_k$: contiguity is the row-major choice of those coefficients.
- **Rule.** A tensor is **contiguous** (row-major, C order) when $s_k=\prod_{j\gt k}n_j$.
- **Formula.** $\text{off}=\sum_k i_k\prod_{j\gt k}n_j$; for `(M,N)` this gives stride `(N,1)`, and for `(B,M,N)` it gives `(M*N,N,1)`.
- **Example.** For `A = torch.arange(12).reshape(3,4)`, stride is `(4,1)` and the offsets are `[[0,1,2,3],[4,5,6,7],[8,9,10,11]]`; the next code cell asserts the table.
- **Where.** A contiguous tensor is addressed by one flat index, which enables coalesced, vectorized loads.
- **Triton.** If the host calls `.contiguous()`, the kernel may use `offs = 0..numel-1`; check `x.is_contiguous()` first.
- **Failure.** Assuming contiguity after `t()`, slicing, `expand()`, or `permute()` breaks the flat-index shortcut.
- **Bridge.** §4.7 writes the row-major formula explicitly.

### 4.7 Row-major memory

- **Connection.** Builds on §4.6's contiguous rule as the default layout.
- **Rule.** The last axis is fastest: one column step is one element, one row step is `N` elements.
- **Formula.** $\text{off}(m,n)=mN+n$ with stride `(N,1)`.
- **Example.** For the `(3,4)` matrix, row `m` starts at offset `4m`, and the jump between rows is $4\cdot4=16$ bytes in `float32`; the next code cell asserts the offset table.
- **Where.** This is PyTorch's default and the layout assumed by most kernels, including the 2-D kernel in Part III.
- **Triton.**
```python
offs = rows[:, None] * N + cols[None, :]
```
- **Failure.** `rows * M + cols` is column-major and reads the wrong element whenever `M != N`.
- **Bridge.** §4.8 takes the other choice: first axis fastest.

### 4.8 Column-major memory

- **Connection.** Builds on §4.7 by swapping the fast axis; a transpose is the common source.
- **Rule.** The first axis is fastest: one row step is one element, one column step is `M` elements.
- **Formula.** $\text{off}(m,n)=m+nM$ with stride `(1,M)`.
- **Example.** For the same logical `(3,4)` matrix, offsets are `[[0,3,6,9],[1,4,7,10],[2,5,8,11]]` and storage order is `[0,4,8,1,5,9,2,6,10,3,7,11]`; the next code cell asserts both.
- **Where.** This is Fortran order and the layout of a transposed row-major tensor.
- **Triton.** Pass `x.stride(0)` and `x.stride(1)` as runtime arguments; a hard-coded `N` cannot read column-major.
- **Failure.** Passing a column-major tensor to a row-major kernel hides on square matrices; test a rectangular case.
- **Bridge.** §4.9 adds `itemsize` so strides become byte distances.

In [ ]:
def test_row_col_offsets() -> None:
    rows = torch.arange(3)[:, None]
    cols = torch.arange(4)[None, :]
    row_offsets = rows * 4 + cols
    col_offsets = rows * 1 + cols * 3

    A_row = torch.arange(12).reshape(3, 4)
    A_col = A_row.t().contiguous().t()

    assert torch.equal(row_offsets,
                       torch.tensor([[0, 1, 2, 3], [4, 5, 6, 7], [8, 9, 10, 11]]))
    assert torch.equal(col_offsets,
                       torch.tensor([[0, 3, 6, 9], [1, 4, 7, 10], [2, 5, 8, 11]]))

    assert tuple(A_row.shape) == (3, 4)
    assert A_row.stride() == (4, 1)
    assert A_row.is_contiguous()
    assert tuple(A_col.shape) == (3, 4)
    assert A_col.stride() == (1, 3)
    assert not A_col.is_contiguous()
    assert torch.equal(A_row, A_col)
    assert A_row.reshape(-1).tolist() == [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
    assert A_col.t().reshape(-1).tolist() == [0, 4, 8, 1, 5, 9, 2, 6, 10, 3, 7, 11]
    assert 4 * 4 == 16  # row jump in bytes for float32
    print("row/column offset tables asserted")


test_row_col_offsets()

### 4.9 Element size and dtype

- **Connection.** Builds on §4.7–§4.8: element strides become byte strides through the itemsize.
- **Rule.** The itemsize is `torch.Tensor.element_size()`; it belongs to the dtype, not the shape.
- **Formula.** $\text{addr}_\text{byte}=\text{base}_\text{byte}+\text{itemsize}(\text{dtype})\cdot\text{off}_\text{elem}$.

| dtype | float64 | float32 | float16 | bfloat16 | int32 | int8 |
|---|---:|---:|---:|---:|---:|---:|
| itemsize | 8 | 4 | 2 | 2 | 4 | 1 |

- **Example.** Element offset `11` is `44` bytes in `float32`, `22` in `float16`, and `11` in `int8`; the next code cell asserts these values.
- **Where.** Byte traffic, alignment, vectorized load width, and cache-line use all depend on itemsize.
- **Triton.** The pointer type carries the dtype, so `x_ptr + off` scales by the element type; reinterpreting the pointer requires reinterpreting the offset.
- **Failure.** Recasting `float32*` to `int8*` and keeping element offsets addresses one byte instead of four.
- **Bridge.** §4.10 reads the full stride vector from PyTorch.

In [ ]:
dtypes = (
    torch.float64, torch.float32, torch.float16, torch.bfloat16,
    torch.int64, torch.int32, torch.int8,
)
expected_itemsize = {
    torch.float64: 8,
    torch.float32: 4,
    torch.float16: 2,
    torch.bfloat16: 2,
    torch.int64: 8,
    torch.int32: 4,
    torch.int8: 1,
}
for dt, itemsize in expected_itemsize.items():
    assert torch.empty(0, dtype=dt).element_size() == itemsize

assert 11 * torch.empty(0, dtype=torch.float32).element_size() == 44
assert 11 * torch.empty(0, dtype=torch.float16).element_size() == 22
assert 11 * torch.empty(0, dtype=torch.int8).element_size() == 11

### 4.10 Tensor strides

- **Connection.** Builds on §4.9's itemsize and §4.6–§4.8's layouts: now read the layout from the tensor itself.
- **Rule.** PyTorch reports `stride(k)` and `storage_offset()` in **elements**; NumPy's `ndarray.strides` are in bytes.
- **Formula.** $\text{off}=\text{storage\_offset}+\sum_k i_k\,\text{stride}(k)$.

| view | shape | stride | storage_offset | contiguous |
|---|---|---|---|---|
| `x` | `(4,6)` | `(6,1)` | `0` | yes |
| `x.t()` | `(6,4)` | `(1,6)` | `0` | no |
| `x[:, 1:4]` | `(4,3)` | `(6,1)` | `1` | no |
| `x[::2]` | `(2,6)` | `(12,1)` | `0` | no |
| `torch.arange(6).expand(4,6)` | `(4,6)` | `(0,1)` | `0` | no |

- **Example.** The table starts from `x = torch.arange(24).reshape(4,6)`; the next code cell asserts every row.
- **Where.** Strides plus `storage_offset` are the complete description of a view.
- **Triton.** Read `x.stride(0)`, `x.stride(1)`, ... on the host and pass them as runtime arguments.
- **Failure.** Using NumPy byte strides where Triton expects elements, or forgetting `storage_offset` in an absolute index.
- **Bridge.** §4.11 applies the same sum to three dimensions.

In [ ]:
def test_stride_views() -> None:
    x = torch.arange(24).reshape(4, 6)
    transpose = x.t()
    sliced = x[:, 1:4]
    stepped = x[::2]
    bias = torch.arange(6).expand(4, 6)

    assert tuple(x.shape) == (4, 6)
    assert x.stride() == (6, 1)
    assert x.storage_offset() == 0
    assert x.is_contiguous()

    assert tuple(transpose.shape) == (6, 4)
    assert transpose.stride() == (1, 6)
    assert transpose.storage_offset() == 0
    assert not transpose.is_contiguous()

    assert tuple(sliced.shape) == (4, 3)
    assert sliced.stride() == (6, 1)
    assert sliced.storage_offset() == 1
    assert not sliced.is_contiguous()

    assert tuple(stepped.shape) == (2, 6)
    assert stepped.stride() == (12, 1)
    assert stepped.storage_offset() == 0
    assert not stepped.is_contiguous()

    assert tuple(bias.shape) == (4, 6)
    assert bias.stride() == (0, 1)
    assert bias.storage_offset() == 0
    assert not bias.is_contiguous()
    print("stride-view table asserted")


test_stride_views()

### 4.11 Multidimensional address calculation

- **Connection.** Builds on §4.10's general stride sum and adds one term per axis.
- **Rule.** For a contiguous `(B,M,N)` tensor, $(s_0,s_1,s_2)=(MN,N,1)$; for a view, read the three strides from PyTorch.
- **Formula.** $\text{off}(b,m,n)=b s_0+m s_1+n s_2$ with $s_k=\text{stride}(k)$.

| tensor | shape | stride | index | offset | value |
|---|---|---|---|---|---|
| `A = arange(12).reshape(3,4)` | `(3,4)` | `(4,1)` | `(2,3)` | `2*4+3*1 = 11` | `11` |
| `B = arange(24).reshape(2,3,4)` | `(2,3,4)` | `(12,4,1)` | `(1,2,3)` | `1*12+2*4+3*1 = 23` | `23` |
| `C = arange(120).reshape(2,3,4,5)` | `(2,3,4,5)` | `(60,20,5,1)` | `(1,2,3,4)` | `1*60+2*20+3*5+4*1 = 119` | `119` |

- **Example.** The next code cell asserts every shape, stride, offset, and value in the table.
- **Where.** This is the map used by every 2-D and 3-D tiled kernel in the book.
- **Triton.**
```python
offs = b_idx[:, None, None] * s0 + m_idx[None, :, None] * s1 + n_idx[None, None, :] * s2
```
- **Failure.** Omitting the batch term gives an address correct only for `b=0`; the error grows with `b`.
- **Bridge.** §4.12 lets all strides come from a non-contiguous view.

In [ ]:
A = torch.arange(12).reshape(3, 4)
B = torch.arange(24).reshape(2, 3, 4)
C = torch.arange(120).reshape(2, 3, 4, 5)

assert tuple(A.shape) == (3, 4) and A.stride() == (4, 1)
assert 2 * A.stride(0) + 3 * A.stride(1) == 11
assert A[2, 3].item() == A.reshape(-1)[11].item() == 11

assert tuple(B.shape) == (2, 3, 4) and B.stride() == (12, 4, 1)
assert 1 * B.stride(0) + 2 * B.stride(1) + 3 * B.stride(2) == 23
assert B[1, 2, 3].item() == B.reshape(-1)[23].item() == 23

assert tuple(C.shape) == (2, 3, 4, 5) and C.stride() == (60, 20, 5, 1)
assert (1 * C.stride(0) + 2 * C.stride(1)
        + 3 * C.stride(2) + 4 * C.stride(3)) == 119
assert C[1, 2, 3, 4].item() == C.reshape(-1)[119].item() == 119

### 4.12 Pointer arithmetic for non-contiguous tensors

- **Connection.** Builds on §4.10–§4.11: the stride sum now comes from a view, not from the shape.
- **Rule.** For a non-contiguous view, the strides are not the row-major products; pass them as runtime arguments and use the general affine formula.
- **Formula.** $\text{off}(i,j)=i s_0+j s_1$ with $s_0=\text{stride}(0)$ and $s_1=\text{stride}(1)$.

| view | shape | stride | storage_offset | `data_ptr` delta | index | offset | storage index | value |
|---|---|---|---|---|---|---|---|---|
| `A.t()` | `(4,3)` | `(1,4)` | `0` | `0` bytes | `(3,2)` | `3*1+2*4 = 11` | `11` | `11` |
| `A[:, 1:3]` | `(3,2)` | `(4,1)` | `1` | `4` bytes | `(2,1)` | `2*4+1*1 = 9` | `1+9 = 10` | `10` |

- **Example.** The table uses `A = torch.arange(12, dtype=torch.float32).reshape(3,4)`; the next code cell asserts every entry.
- **Where.** Feeding a transposed or sliced tensor directly to a stride-aware kernel avoids a materializing `.contiguous()` copy.
- **Triton.**
```python
offs = rows[:, None] * stride_m + cols[None, :] * stride_n
x = tl.load(x_ptr + offs, mask=mask, other=0.0)
```
- **Failure.** Passing only the shape and assuming `(N,1)` reads the transpose of the intended tile with no out-of-bounds warning.
- **Bridge.** Part II turns this address chain into an explicit Python gather.
- **Where this leads.** Chapter 5 — Loading and Storing Data turns these addresses into masked loads and stores.

In [ ]:
A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
At = A.t()
S = A[:, 1:3]

assert tuple(At.shape) == (4, 3)
assert At.stride() == (1, 4)
assert At.storage_offset() == 0
assert not At.is_contiguous()
assert At.data_ptr() - A.data_ptr() == 0

assert tuple(S.shape) == (3, 2)
assert S.stride() == (4, 1)
assert S.storage_offset() == 1
assert not S.is_contiguous()
assert S.data_ptr() - A.data_ptr() == 4

assert 3 * At.stride(0) + 2 * At.stride(1) == 11
assert A.reshape(-1)[11].item() == 11.0
assert 2 * S.stride(0) + 1 * S.stride(1) == 9
assert S.storage_offset() + 9 == 10
assert S[2, 1].item() == A.reshape(-1)[10].item() == 10.0
assert S[0, 0].item() == 1.0

## Chapter map

- **Thread.** Each row is one link in the chain from a logical index to a pointer expression.
- **Reading.** The key formula is the arithmetic; the shape/stride rule is the host contract; the last column names the kernel or demo.
- **Bridge.** The worked example applies the table to a single `(3,4)` matrix.

| § | Topic | Key formula | Shape/stride rule | Kernel/demo |
|---|---|---|---|---|
| **4.1** | Pointer arguments | $\text{addr}=\text{base}+\text{off}$ | offset is relative to the view's `data_ptr` | 1-D `+1` kernel |
| **4.2** | Base pointer | `base = x_ptr` | `data_ptr` already includes `storage_offset` | worked example |
| **4.3** | Pointer plus offset | `x_ptr + offs` | `offs` has the tile shape | 1-D and 2-D kernels |
| **4.4** | Element offset vs byte offset | $\text{addr}_\text{byte}=\text{base}_\text{byte}+b\cdot\text{off}$ | `b = itemsize(dtype)` | offset table |
| **4.5** | Pointer arithmetic | $\text{base}+\sum_k i_k s_k$ | one affine map per element | 1-D kernel |
| **4.6** | Contiguous memory | $s_k=\prod_{j\gt k}n_j$ | `is_contiguous()` is true | 2-D row-major kernel |
| **4.7** | Row-major memory | $\text{off}(i,j)=iN+j$ | stride `(N, 1)` | 2-D kernel |
| **4.8** | Column-major memory | $\text{off}(i,j)=i+jM$ | stride `(1, M)` | transpose demo |
| **4.9** | Element size and dtype | $\text{addr}_\text{byte}=b\cdot\text{off}$ | fp32 4 B, fp16 2 B, int8 1 B | itemsize table |
| **4.10** | Tensor strides | $\text{off}=\sum_k i_k s_k$ | PyTorch strides are in **elements** | view table |
| **4.11** | Multidimensional address calculation | $\text{off}(b,i,j)=b s_0+i s_1+j s_2$ | one stride per axis | 3-D numeric check |
| **4.12** | Pointer arithmetic for non-contiguous tensors | $\text{off}=i s_0+j s_1$ | pass `stride()` at runtime | stride-aware kernel |
| **★** | Capstone | $\text{src}=i s_0+j s_1$ | strided source, contiguous destination | panel copy |

## Worked numerical example

- **Connection.** Applies §4.5–§4.12 to one matrix and checks every value in the next code cell.
- **Matrix.** Take

$$
A=\begin{bmatrix}
0&1&2&3\\
4&5&6&7\\
8&9&10&11
\end{bmatrix}\in\mathbb R^{3\times4},
\qquad
\text{shape}=(3,4),\quad
\text{stride}=(4,1),\quad
\text{storage\_offset}=0.
$$

- **Address of `(2,3)`.** For `float32`, `itemsize = 4`; $\text{off}(2,3)=2\cdot4+3\cdot1=11$, so the byte offset is $11\cdot4=44$, and `A[2,3] = A.reshape(-1)[11] = 11`.
- **Element vs byte.** The address table is:

| quantity | element units | byte units |
|---|---:|---:|
| address of `(2,3)` | `11` | `44` |
| row jump from `(1,3)` to `(2,3)` | `4` | `16` |

- **Transpose.** $A^\top$ has shape `(4,3)` and stride `(1,4)`; its index `(3,2)` maps to $3\cdot1+2\cdot4=11$, the same storage element and the same `11` value.
- **Slice.** $S=A[:,1{:}3]$ has shape `(3,2)`, stride `(4,1)`, and `storage_offset = 1`; the view-relative offset of `S[2,1]` is $2\cdot4+1\cdot1=9$, the storage index is $1+9=10$, and `S[2,1] = A.reshape(-1)[10] = 10`.
- **Views.** The complete table is:

| view | shape | stride | storage_offset | an index | offset | value |
|---|---|---|---|---|---|---|
| `A` | `(3,4)` | `(4,1)` | `0` | `(2,3)` | `11` | `11` |
| `A.t()` | `(4,3)` | `(1,4)` | `0` | `(3,2)` | `11` | `11` |
| `A[:, 1:3]` | `(3,2)` | `(4,1)` | `1` | `(2,1)` | `9` (storage `10`) | `10` |
| `A.t().contiguous()` | `(4,3)` | `(3,1)` | `0` | `(3,2)` | `3*3+2*1 = 11` | `11` |

- **Copy.** A `contiguous()` copy of $A^\top$ has shape `(4,3)` and stride `(3,1)`: values are unchanged, `data_ptr` changes, and storage order is rearranged.
- **Bridge.** The next code cell asserts every entry in these tables.

In [ ]:
def assert_worked_example() -> None:
    A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
    item = A.element_size()
    base = A.data_ptr()

    assert tuple(A.shape) == (3, 4)
    assert A.stride() == (4, 1)
    assert A.storage_offset() == 0
    assert item == 4
    assert base == A.data_ptr()

    elem_off = 2 * A.stride(0) + 3 * A.stride(1)
    byte_off = elem_off * item
    assert elem_off == 11
    assert byte_off == 44
    assert A[2, 3].item() == A.reshape(-1)[11].item() == 11.0
    assert base + byte_off == A[2, 3].data_ptr()
    assert 4 * item == 16

    At = A.t()
    assert tuple(At.shape) == (4, 3)
    assert At.stride() == (1, 4)
    assert At.storage_offset() == 0
    assert not At.is_contiguous()
    assert 3 * At.stride(0) + 2 * At.stride(1) == 11
    assert At[3, 2].item() == 11.0

    S = A[:, 1:3]
    assert tuple(S.shape) == (3, 2)
    assert S.stride() == (4, 1)
    assert S.storage_offset() == 1
    assert not S.is_contiguous()
    assert 2 * S.stride(0) + 1 * S.stride(1) == 9
    assert S.storage_offset() + 9 == 10
    assert S[2, 1].item() == A.reshape(-1)[10].item() == 10.0
    assert S.data_ptr() - A.data_ptr() == 4

    Atc = At.contiguous()
    assert tuple(Atc.shape) == (4, 3)
    assert Atc.stride() == (3, 1)
    assert Atc.storage_offset() == 0
    assert Atc.is_contiguous()
    assert 3 * Atc.stride(0) + 2 * Atc.stride(1) == 11
    assert Atc[3, 2].item() == 11.0
    assert torch.equal(Atc, At)
    assert Atc.data_ptr() != At.data_ptr()


assert_worked_example()

In [ ]:
def test_worked_example() -> None:
    assert_worked_example()
    print("worked example: addresses, strides, and copy asserted")


test_worked_example()

## Part II — Method 01: host-side address arithmetic

- **Connection.** Builds on §4.5–§4.12; the same address formula is now executed in Python.
- **Rule.** Loop over flat output indices, reconstruct the multi-index, compute the storage index, and gather the value.
- **Formula.** $\text{storage index}=\text{storage\_offset}+\sum_k i_k\,\text{stride}(k)$.
- **Example.**
```text
flat output index -> multi-index -> storage index -> value
```
- **Where.** `as_strided` only provides a flat view of the underlying storage; the gather itself is driven by the offset formula, not by PyTorch indexing.
- **Triton.** On CPU this is fine for small tensors; on a GPU it is one Python iteration per element, which is why Method 02 exists.
- **Bridge.** Part III replaces the Python loop with `tl.arange` and one offset tile.

In [ ]:
def naive_strided_copy(x: torch.Tensor) -> torch.Tensor:
    '''Method 01: gather with explicit storage_offset and stride().'''
    shape = x.shape
    strides = x.stride()
    base = x.storage_offset()
    storage_len = x.untyped_storage().nbytes() // x.element_size()
    storage = x.as_strided((storage_len,), (1,), 0)
    out = torch.empty(shape, dtype=x.dtype)

    for flat in range(x.numel()):
        idx = []
        rem = flat
        for size in reversed(shape):
            idx.append(rem % size)
            rem //= size
        idx = tuple(reversed(idx))
        off = base + sum(i * s for i, s in zip(idx, strides))
        out[idx] = storage[off]
    return out


def naive_element_address(x: torch.Tensor, idx: tuple) -> int:
    '''Storage index of a logical index, including storage_offset.'''
    return x.storage_offset() + sum(i * s for i, s in zip(idx, x.stride()))


A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
assert naive_element_address(A.t(), (3, 2)) == 11
assert naive_element_address(A[:, 1:3], (2, 1)) == 10
assert torch.equal(naive_strided_copy(A.t()), A.t().contiguous())

In [ ]:
def test_naive_strided() -> None:
    A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
    for name, view in (
        ("contiguous", A),
        ("transpose", A.t()),
        ("slice", A[:, 1:3]),
        ("step", A[::2]),
    ):
        got = naive_strided_copy(view)
        want = view.contiguous()
        assert torch.equal(got, want), name
        assert got.stride() == want.stride(), name

    assert naive_element_address(A.t(), (3, 2)) == 11
    assert naive_element_address(A[:, 1:3], (2, 1)) == 10
    print("method 01: explicit-stride gather matches contiguous()")


test_naive_strided()

## Part III — Method 02: Triton kernels with element offsets

- **Connection.** Builds on Part II and §4.3: the loop becomes a tile, and the address becomes `base + offs`.
- **Rule.** Each program owns `BLOCK` consecutive elements; the tail is guarded by a mask.
- **Formula.** $\text{offs}=\text{pid}\cdot B+\{0,\ldots,B-1\}$, $\text{mask}=\text{offs}\lt N$, and $P=\lceil N/B\rceil$.
- **Example.** The 2-D kernel uses $\text{offs}(i,j)=iN+j$ with $i=\text{pid}_0 B_M+\{0,\ldots,B_M-1\}$ and $j=\text{pid}_1 B_N+\{0,\ldots,B_N-1\}$.
- **Offset dtype.** `tl.arange` and `tl.program_id` produce 32-bit integers, so the largest offset must satisfy $\text{numel}-1\le2^{31}-1$; for larger tensors cast `offs` to `tl.int64`.
- **Where.** The test runs both the int32 and int64 paths on a small tensor; values are identical and only the offset width changes.
- **Triton.** The next code cell asserts the int32 and int64 limits before the kernels run.
- **Bridge.** Part IV stops assuming that the input is contiguous.

In [ ]:
@triton.jit
def m4_add_one_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)          # int32 by default
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x + 1.0, mask=mask)


@triton.jit
def m4_add_one_i64_kernel(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    offs = offs.to(tl.int64)                          # for numel > 2**31
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x + 1.0, mask=mask)


def add_one_1d(x: torch.Tensor, BLOCK: int = 256, use_i64: bool = False) -> torch.Tensor:
    '''Method 02: pointer + element offsets on a contiguous 1-D tensor.'''
    x = x.contiguous()
    out = torch.empty_like(x)
    kernel = m4_add_one_i64_kernel if use_i64 else m4_add_one_kernel
    kernel[(triton.cdiv(x.numel(), BLOCK),)](x, out, x.numel(), BLOCK=BLOCK)
    return out


assert torch.iinfo(torch.int32).max == 2**31 - 1
assert 2**31 == 2_147_483_648
assert torch.iinfo(torch.int64).max == 9_223_372_036_854_775_807

In [ ]:
@triton.jit
def m4_scale_2d_kernel(x_ptr, out_ptr, M, N, BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    # Row-major address: offset = i*N + j.
    offs = rows[:, None] * N + cols[None, :]
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x * 2.0, mask=mask)


def scale_2d(x: torch.Tensor, BLOCK_M: int = 16, BLOCK_N: int = 16) -> torch.Tensor:
    '''Method 02: 2-D row-major offsets; input is made contiguous.'''
    x = x.contiguous()
    M, N = x.shape
    out = torch.empty_like(x)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    m4_scale_2d_kernel[grid](x, out, M, N, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out


def test_method02() -> None:
    require_gpu("m4 method 02")
    x = torch.arange(7, device=DEVICE, dtype=torch.float32)
    check("1-D element offsets", add_one_1d(x, BLOCK=4), x + 1)
    check("1-D int64 offsets", add_one_1d(x, BLOCK=4, use_i64=True), x + 1)

    A = torch.arange(12, device=DEVICE, dtype=torch.float32).reshape(3, 4)
    check("2-D row-major offsets", scale_2d(A, BLOCK_M=2, BLOCK_N=4), A * 2)
    print("method 02: 1-D int32/int64 and 2-D row-major kernels")

## Part IV — Method 03: stride-aware kernels for non-contiguous views

- **Connection.** Builds on Part III and §4.12: the kernel now receives the view's strides instead of assuming `(N,1)`.
- **Rule.** Pass four runtime strides: two for the input view and two for the output.
- **Formula.** $\text{src}(i,j)=i s^{(x)}_0+j s^{(x)}_1$ and $\text{dst}(i,j)=i s^{(o)}_0+j s^{(o)}_1$.
- **Example.** For an `(M,N)` tensor with `b` bytes per element, the stride-aware path moves $2MNb$ bytes; the materializing path moves $4MNb$.
- **Where.** For `M=1024`, `N=2048`, and `b=4`, those are `16,777,216` bytes (`16.78 MB`) and `33,554,432` bytes (`33.55 MB`), a factor of two; the next code cell asserts the exact counts.
- **Triton.** The copy path can still be faster when the strided access is badly uncoalesced, but its traffic is twice as large.
- **Failure.** `data_ptr` already includes `storage_offset`; do not add it again inside the kernel.
- **Bridge.** The applied example fixes the output stride to a panel width.

In [ ]:
@triton.jit
def m4_strided_add_one_kernel(
    x_ptr, out_ptr, M, N,
    stride_m, stride_n, out_stride_m, out_stride_n,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    src = rows[:, None].to(tl.int64) * stride_m + cols[None, :].to(tl.int64) * stride_n
    dst = rows[:, None].to(tl.int64) * out_stride_m + cols[None, :].to(tl.int64) * out_stride_n

    x = tl.load(x_ptr + src, mask=mask, other=0.0)
    tl.store(out_ptr + dst, x + 1.0, mask=mask)


def strided_add_one(x: torch.Tensor, BLOCK_M: int = 16, BLOCK_N: int = 16) -> torch.Tensor:
    '''Method 03: read any 2-D view through its explicit strides.'''
    assert x.ndim == 2, "expected a 2-D tensor"
    M, N = x.shape
    out = torch.empty((M, N), dtype=x.dtype, device=x.device)  # contiguous
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    m4_strided_add_one_kernel[grid](
        x, out, M, N,
        x.stride(0), x.stride(1), out.stride(0), out.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
    )
    return out

In [ ]:
A = torch.arange(12, dtype=torch.float32).reshape(3, 4)
At = A.t()
Ac = At.contiguous()

assert tuple(At.shape) == (4, 3)
assert At.stride() == (1, 4)
assert not At.is_contiguous()
assert tuple(Ac.shape) == (4, 3)
assert Ac.stride() == (3, 1)
assert Ac.is_contiguous()
assert torch.equal(At, Ac)
assert At.data_ptr() != Ac.data_ptr()

M, N, b = 1024, 2048, 4
strided = 2 * M * N * b
copied = 4 * M * N * b
assert strided == 16_777_216
assert copied == 33_554_432
assert round(strided / 1e6, 2) == 16.78
assert round(copied / 1e6, 2) == 33.55
assert copied == 2 * strided

In [ ]:
def test_method03() -> None:
    require_gpu("m4 method 03")
    A = torch.arange(12, device=DEVICE, dtype=torch.float32).reshape(3, 4)

    check("contiguous input", strided_add_one(A, BLOCK_M=2, BLOCK_N=2), A + 1)

    At = A.t()
    check("transposed input", strided_add_one(At, BLOCK_M=2, BLOCK_N=2), At + 1)

    S = A[:, 1:3]
    check("sliced input", strided_add_one(S, BLOCK_M=2, BLOCK_N=2), S + 1)

    Ac = At.contiguous()
    check("materialized copy", strided_add_one(Ac, BLOCK_M=2, BLOCK_N=2), Ac + 1)
    print("method 03: transposed and sliced strides match torch")

## Applied example — copy a strided column panel

- **Connection.** Builds on Part IV: the source is a column panel with arbitrary source strides.
- **Rule.** Extract $P=A[:,c_0{:}c_1]$ with stride `(N,1)` and `storage_offset = c_0`, then write a contiguous destination.
- **Formula.** $\text{src}(i,j)=\text{base}+iN+(c_0+j)$ and $\text{dst}(i,j)=iW+j$, where $W=c_1-c_0$.
- **Example.** For `A = torch.arange(20).reshape(4,5)` and `A[:,1:4]`, the panel has shape `(4,3)`, stride `(5,1)`, and `storage_offset = 1`; `A.t()` has shape `(5,4)` and stride `(1,5)`.
- **Where.** The panel is non-contiguous whenever its width is less than `N`, so a downstream dense kernel needs a copy.
- **Triton.** The kernel reads with `stride_m`/`stride_n` and writes with `N` as the row stride.
- **Failure.** Treating the panel as contiguous reads the wrong column; the test asserts the panel shapes and strides before checking values.
- **Bridge.** Chapter 5 turns these addresses into masked loads and stores.

In [ ]:
@triton.jit
def m4_panel_kernel(
    x_ptr, out_ptr, M, N,
    stride_m, stride_n,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rows[:, None] < M) & (cols[None, :] < N)

    src = rows[:, None].to(tl.int64) * stride_m + cols[None, :].to(tl.int64) * stride_n
    dst = rows[:, None].to(tl.int64) * N + cols[None, :].to(tl.int64)

    x = tl.load(x_ptr + src, mask=mask, other=0.0)
    tl.store(out_ptr + dst, x, mask=mask)


def copy_panel(x: torch.Tensor, BLOCK_M: int = 16, BLOCK_N: int = 16) -> torch.Tensor:
    '''Copy any 2-D view into a contiguous buffer using source strides.'''
    assert x.ndim == 2, "expected a 2-D tensor"
    M, N = x.shape
    out = torch.empty((M, N), dtype=x.dtype, device=x.device)
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
    m4_panel_kernel[grid](
        x, out, M, N, x.stride(0), x.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
    )
    return out


A_cpu = torch.arange(20).reshape(4, 5)
panel_cpu = A_cpu[:, 1:4]
assert tuple(panel_cpu.shape) == (4, 3)
assert panel_cpu.stride() == (5, 1)
assert panel_cpu.storage_offset() == 1
assert tuple(A_cpu.t().shape) == (5, 4)
assert A_cpu.t().stride() == (1, 5)


def test_panel() -> None:
    require_gpu("m4 strided panel")
    A = torch.arange(20, device=DEVICE, dtype=torch.float32).reshape(4, 5)

    panel = A[:, 1:4]
    assert tuple(panel.shape) == (4, 3)
    assert panel.stride() == (5, 1)
    assert panel.storage_offset() == 1
    check("column panel", copy_panel(panel, BLOCK_M=2, BLOCK_N=2), panel.contiguous())

    transposed = A.t()
    assert tuple(transposed.shape) == (5, 4)
    assert transposed.stride() == (1, 5)
    check("transposed panel", copy_panel(transposed, BLOCK_M=2, BLOCK_N=2),
          transposed.contiguous())
    print("applied: column panel and transpose copy")

## Pitfalls

- **Connection.** Collects the failure modes from §4.1–§4.12 and the Method 01–03 kernels.

| Pitfall | What happens | Fix |
|---|---|---|
| Element vs byte offset | a byte offset added to `x_ptr` skips that many elements | keep Triton offsets in elements; convert with `itemsize` only for diagnostics |
| Wrong stride | a `(M,N)` tensor is read with `(N,1)` when its actual stride is `(1,M)` | read `x.stride()` and pass it, or use the general formula |
| Transposed view | values come back transposed with no out-of-bounds error | test a rectangular transpose where `M != N` |
| `int32` overflow for `numel > 2**31` | offset wraps negative and the load faults or reads the wrong tile | cast `offs` to `tl.int64` before `x_ptr + offs` |
| Assuming contiguous | `t()`, slicing, `expand()`, and `permute()` change the stride | check `is_contiguous()`; call `.contiguous()` only when you mean to copy |
| Negative strides | Triton pointer arithmetic does not validate bounds; a negative step can walk before the base | materialize with `contiguous()` or use a positive-stride view |
| Forgetting `storage_offset` | an absolute storage index is off by the view's offset | use `data_ptr()` as the base, or add `storage_offset` in host-side arithmetic |
| NumPy strides in bytes | offsets are `itemsize` times too large | divide NumPy byte strides by `itemsize` when passing them to Triton |
| Zero-stride `expand` | every row reads or writes the same element | materialize, or guard the store so each destination is written once |

## Self-check

- For a `(3,4)` `float32` matrix with stride `(4,1)`, what are the element and byte offsets of `(2,3)`?
- Why is `x_ptr + 44` wrong when you meant byte offset `44`? What is the correct expression?
- Give the row-major and column-major offset formulas for a `(3,4)` matrix and evaluate both at `(2,3)`.
- What are the shape, stride, `storage_offset`, and `is_contiguous()` of `A.t()` and `A[:, 1:3]`?
- Why does `data_ptr()` already include `storage_offset`, and what happens if a kernel adds it again?
- At what element count does a 32-bit offset become unsafe, and how do you fix it?
- Compare the byte traffic of a stride-aware kernel with a `.contiguous()` plus kernel path for `(M,N)` `float32`.
- A kernel hard-codes `offs = i*N + j` and receives a transposed tensor. Describe the wrong values it reads when `M != N`.
- Why is `bias.expand(4,6)` not contiguous, and why is its stride `(0,1)`?
- Does `contiguous()` change the values of a view? Does it change the stride and `data_ptr`?

## Summary

- A kernel argument such as `x_ptr` is a typed base pointer; `x_ptr + off` advances by `off` **elements**.
- The byte address is `base_byte + itemsize*off`; PyTorch's `data_ptr()` and NumPy's strides are byte quantities.
- Row-major memory uses `off = i*N + j` and stride `(N,1)`; column-major uses `off = i + j*M` and stride `(1,M)`.
- A tensor's strides, plus its `storage_offset`, completely describe the address map of a view.
- `data_ptr()` already includes `storage_offset`, so kernel offset `0` is the view's first logical element.
- Non-contiguous tensors need explicit strides; passing them avoids a copy but may produce uncoalesced access.
- `tl.arange` and `tl.program_id` produce int32 offsets; cast to `tl.int64` when the tensor can exceed `2**31` elements.
- A stride-aware kernel and a contiguous-copy path compute the same values but move different numbers of bytes.

**Where this leads.** Chapter 5 — Loading and Storing Data turns these pointer addresses into masked loads, stores, and boundary checks.

**Next:** [Chapter 5 — Loading and Storing Data](../ch05_load_store/05_complete.ipynb)

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [← Previous](../../part1_foundations/ch03_vector_construction/03_complete.ipynb)

# Exercises

- **Connection.** These exercises revisit the address chain from §4.1–§4.12 and the Method 01–03 kernels.

1. **Address by hand.** For a `(5,7)` `float32` matrix, compute the element and byte offsets of `(4,6)`. Verify with `A.reshape(-1)[...]` and `A.data_ptr()`.
2. **Element vs byte.** Write a function `byte_address(x, idx)` that returns `x.data_ptr() + itemsize*offset`. Compare it with `x[idx].data_ptr()` for three indices.
3. **Column-major.** Build a column-major `(3,4)` view with stride `(1,3)` and write the offset formula. Check `A_col.t().reshape(-1)` against the hand-written storage order.
4. **Strides.** Create a `(4,6)` tensor, then inspect shape, stride, and `storage_offset` for `t()`, `[:, 2:5]`, `[::2]`, and `expand(4,6)`. Explain each stride.
5. **1-D kernel.** Modify `m4_add_one_kernel` to compute `2*x + 1`. Test `N=7` with `BLOCK=4` and confirm the tail mask.
6. **int64.** Make a kernel that computes offsets as `tl.int64` and expose the offset dtype with a compile-time branch. Why is int32 enough for most tensors but not all?
7. **2-D kernel.** Change `m4_scale_2d_kernel` to write `x + 10` and test a `(5,3)` matrix with `BLOCK_M=2`, `BLOCK_N=2`.
8. **Stride-aware.** Run `strided_add_one` on a `(4,6)` tensor, its transpose, and `x[:, 1:4]`. Compare against `.contiguous() + 1`.
9. **Byte traffic.** Derive the read/write counts for the stride-aware and contiguous-copy paths for `(1024,2048)` `float32`. When would the copy path still be faster?
10. **Panel copy.** Generalize `copy_panel` to copy a 3-D view with strides `(s0,s1,s2)` into a contiguous `(B,M,N)` buffer. Which extra stride argument do you need?

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_address_arithmetic()
test_row_col_offsets()
test_stride_views()
test_naive_strided()

print("\nRunning GPU tests (skipped without CUDA/ROCm; the interpreter forces them on) ...")
for fn in (test_method02, test_method03, test_panel):
    run_if_gpu(fn)